# EV-Insight 数据预处理与特征工程
---
本 Notebook 实现论文第 2.3 节的数据预处理全流程。

**执行步骤:**
1. 加载原始数据
2. 文本字段清理 & 品牌名称统一
3. 数值字段标准化
4. 价格字段解析 (如 "14.98-19.98万" → 数值)
5. 缺失值处理 (品牌中位数分层填充)
6. 异常值检测与剔除 (IQR 方法)
7. 衍生特征构建
8. 类别变量编码
9. 保存清洗后数据

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# 设置中文显示
plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

from src.preprocess import preprocess_ev_data, create_derived_features, encode_categorical

print('✅ 依赖库加载完成')

In [ ]:
# 加载原始数据
df_raw = pd.read_csv('../data/ev_data_raw.csv')
print(f'原始数据: {len(df_raw)} 条记录, {len(df_raw.columns)} 个字段')
df_raw.head(10)

In [ ]:
# 查看缺失值情况
missing = df_raw.isna().sum()
missing = missing[missing > 0].sort_values(ascending=False)
print('缺失值统计:')
print(missing)

In [ ]:
# 运行完整预处理流水线
df_clean = preprocess_ev_data(df_raw)
print(f'\n清洗后数据: {len(df_clean)} 条记录, {len(df_clean.columns)} 个特征')

In [ ]:
# 价格分布可视化
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].hist(df_clean['price_median'], bins=50, color='#667eea', edgecolor='white', alpha=0.8)
axes[0].axvline(df_clean['price_median'].median(), color='red', linestyle='--', 
                label=f'中位数: {df_clean["price_median"].median():.1f}万')
axes[0].set_xlabel('价格 (万元)')
axes[0].set_ylabel('车型数')
axes[0].set_title('价格分布直方图')
axes[0].legend()

axes[1].hist(df_clean['range_km'], bins=50, color='#22c55e', edgecolor='white', alpha=0.8)
axes[1].axvline(df_clean['range_km'].median(), color='red', linestyle='--',
                label=f'中位数: {df_clean["range_km"].median():.0f}km')
axes[1].set_xlabel('续航里程 (km)')
axes[1].set_ylabel('车型数')
axes[1].set_title('续航里程分布直方图')
axes[1].legend()

axes[2].scatter(df_clean['range_km'], df_clean['price_median'],
                c=df_clean['battery_capacity'], cmap='viridis', alpha=0.5, s=10)
axes[2].set_xlabel('续航里程 (km)')
axes[2].set_ylabel('价格 (万元)')
axes[2].set_title('续航 vs 价格 (颜色=电池容量)')
cbar = plt.colorbar(axes[2].collections[0], ax=axes[2])
cbar.set_label('电池容量 (kWh)')

plt.tight_layout()
plt.show()

In [ ]:
# 品牌 Top 15
top_brands = df_clean['brand'].value_counts().head(15)
fig, ax = plt.subplots(figsize=(10, 6))
top_brands.sort_values().plot(kind='barh', ax=ax, color='#667eea')
ax.set_xlabel('车型数')
ax.set_title('Top 15 品牌车型数量')
plt.tight_layout()
plt.show()

In [ ]:
# 相关性热力图
numeric_cols = ['price_median', 'range_km', 'battery_capacity', 'power_kw', 
                'torque_nm', 'accel_0_100', 'length_mm', 'wheelbase_mm', 'user_score']
numeric_cols = [c for c in numeric_cols if c in df_clean.columns]

corr_matrix = df_clean[numeric_cols].corr()

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='RdBu_r', 
            center=0, square=True, ax=ax)
ax.set_title('特征相关性热力图')
plt.tight_layout()
plt.show()

In [ ]:
# 保存清洗后的数据
df_clean.to_csv('../data/ev_data_cleaned.csv', index=False, encoding='utf-8-sig')
print('✅ 清洗后数据已保存到: ../data/ev_data_cleaned.csv')